## Scenario: A software company wants to build its first AI assistant. 
### Tasks: Create a basic LLM-powered agent capable of handling user queries, maintaining context, and performing task execution workflows.

In [1]:
import os
import time
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

from langchain_ollama import ChatOllama, OllamaEmbeddings

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate

from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage,
    ToolMessage
)

from langchain.agents import create_agent

from langgraph.checkpoint.memory import InMemorySaver
from langchain_google_genai import ChatGoogleGenerativeAI

print("All libraries imported successfully!")

C:\Users\Sudhanshu Singh\AppData\Local\Temp\ipykernel_19136\3260249849.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


All libraries imported successfully!


In [2]:
# PDF_PATH = "who.pdf"
PDF_PATH = "data/report.pdf"

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

print(f"PDF loaded successfully: {len(documents)} pages")

PDF loaded successfully: 13 pages


In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks: {len(chunks)}")

Total chunks: 124


In [4]:
for i, chunk in enumerate(chunks[:3]):
    print(f"\nChunk {i + 1}")
    print(chunk.page_content)
    print("-" * 80)


Chunk 1
Turbocharging Vector Databases using Modern SSDs
Joobo Shim
Seoul National University
jbshim@snu.ac.kr
Jaewon Oh
Seoul National University
jaewon.oh@snu.ac.kr
Hongchan Roh
Dnotitia
hongchan.roh@dnotitia.com
Jaeyoung Do∗
Seoul National University
jaeyoung.do@snu.ac.kr
Sang-Won Lee
Seoul National University
swlee69@snu.ac.kr
ABSTRACT
Efficient and scalable vector search is critical for modern AI applic-
ations, particularly in retrieval-augmented generation (RAG) and
large-scale semantic search. However, disk-based vector databases
often suffer from significant I/O bottlenecks due to suboptimal
cache hit ratios and inefficient use of modern SSD architectures. In
this work, we introduce a suite of optimizations to enhance the per-
formance of disk-resident Approximate Nearest Neighbor (ANN)
--------------------------------------------------------------------------------

Chunk 2
this work, we introduce a suite of optimizations to enhance the per-
formance of disk-resident Approxi

In [5]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

print("Embedding model initialized successfully!")

Embedding model initialized successfully!


In [6]:
vectors = embeddings.embed_documents(
    [chunk.page_content for chunk in chunks]
)

print(len(vectors), len(vectors[0]))

124 768


In [7]:
vectorstore = Chroma.from_documents(
    chunks,
    embedding=embeddings,
    collection_name="teacher_research_paper",
    persist_directory="./teacher1_chroma_db"
)

In [8]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [15]:
results = retriever.invoke(
    "What are challanges of disk storage?"
)

for doc in results:
    print(doc.page_content)

R. Kadekodi. Diskann: Fast accurate billion-point nearest neighbor search on a
single node. Advances in Neural Information Processing Systems , 32, 2019.
[22] A. Q. Jiang, A. Sablayrolles, A. Mensch, C. Bamford, D. S. Chaplot, D. de las Casas,
F. Bressand, G. Lengyel, G. Lample, L. Saulnier, L. R. Lavaud, M.-A. Lachaux,
P. Stock, T. L. Scao, T. Lavril, T. Wang, T. Lacroix, and W. E. Sayed. Mistral 7b,
2023.
[23] M. Jung and M. T. Kandemir. Sprinkler: Maximizing resource utilization in
many-chip solid state disks. In 2014 IEEE 20th International Symposium on High
Performance Computer Architecture (HPCA), pages 524–535, 2014.
[24] M. Jung, E. H. Wilson, and M. Kandemir. Physically addressed queueing (paq):
Improving parallelism in solid state disks. In 2012 39th Annual International
search. However, pgvector uses sequential blocked I/O during
neighbor-scan, underutilizing SSD parallelism (see Section 2.2).
Moreover, vectors are stored by insertion order, ignoring spatial
locality and gra

In [ ]:

system_prompt = SystemMessage(
    content="""
You are an AI research assistant.

Answer questions using the research paper provided
through the search tool.

If the answer cannot be found in the research paper,
say that you don't know.

Do not make up information that is not supported
by the research paper.
"""
)

memory = InMemorySaver()

In [12]:
'''from langchain.agents import create_agent
agent = create_agent(
    model="ollama:ornith-1.5:9b",
    system_prompt=system_prompt,
    checkpointer=memory
)

print("Agent created")'''

'from langchain.agents import create_agent\nagent = create_agent(\n    model="ollama:ornith-1.5:9b",\n    system_prompt=system_prompt,\n    checkpointer=memory\n)\n\nprint("Agent created")'

In [13]:
'''def search_hospital(query):
    results = retriever.invoke(query)
    return "\n".join(doc.page_content for doc in results)'''

'def search_hospital(query):\n    results = retriever.invoke(query)\n    return "\n".join(doc.page_content for doc in results)'

In [ ]:

from langchain_core.tools import tool

@tool
def search_paper(query: str) -> str:
    """Search the research paper for relevant information."""
    results = retriever.invoke(query)

    return "\n".join(
        doc.page_content
        for doc in results
    )

In [ ]:

llm = ChatOllama(
    model="llama3.2:3b",
    temperature=0
)

agent = create_agent(
    model=llm,
    tools=[search_paper],
    system_prompt=system_prompt,
    checkpointer=memory
)

print("Agent created")

Agent created


In [ ]:

config = {
    "configurable": {
        "thread_id": "research-001"
    }
}

In [ ]:

response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="What are the main challenges faced by disk-based vector databases?"
            )
        ]
    },
    config=config
)
for message in response["messages"]:
    print("\n--- MESSAGE ---")
    print("Type:", type(message).__name__)
    print("Content:", message.content)

    if hasattr(message, "tool_calls"):
        print("Tool calls:", message.tool_calls)


--- MESSAGE ---
Type: HumanMessage
Content: What are the main challenges faced by disk-based vector databases?

--- MESSAGE ---
Type: AIMessage
Content: {"name":"search_paper",""parameters":{"query":"main challenges faced by disk-based vector databases"}}
Tool calls: []
